# Тестовая панель исследовательских экспериментов

Цель этой тетради — не просто прогнать код, а получить **детальный анализ методов** для курсовой.
Поэтому здесь меньше сырых таблиц и больше графиков по трем главным метрикам:
- качество восстановления (`test RMSE`, то есть RMSE на тестовой части),
- среднее время работы,
- среднее число итераций.


In [ ]:
from pathlib import Path
import sys
import importlib

PROJECT_DIR = Path.cwd().resolve()
for candidate in (PROJECT_DIR, *PROJECT_DIR.parents):
    if (candidate / 'synthetic_research_api.py').exists():
        PROJECT_DIR = candidate
        break
else:
    raise FileNotFoundError('Не удалось найти корень проекта: synthetic_research_api.py не найден.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import synthetic_api
import synthetic_research_api

api = importlib.reload(synthetic_api)
research = importlib.reload(synthetic_research_api)

PROJECT_DIR

In [ ]:
METHOD_LABELS = {
    'compact_rgd': 'Compact RGD',
    'compact_rgd_l2': 'Compact RGD + L2',
    'soft_impute': 'Soft-Impute',
    'als': 'ALS',
}

def show_table(rows, n=5):
    table = api.to_dataframe(rows)
    if hasattr(table, 'head'):
        return table.head(n)
    return table[:n]

def prettify_rows(rows):
    pretty = []
    for row in rows:
        updated = dict(row)
        method = updated.get('method')
        if method in METHOD_LABELS:
            updated['method'] = METHOD_LABELS[method]
        pretty.append(updated)
    return pretty

def rows_for_experiment(summary_rows, experiment_name):
    return [row for row in summary_rows if row['experiment_name'] == experiment_name]

def plot_experiment_bundle(
    rows,
    *,
    x,
    xlabel,
    title_base,
    x_tick_rotation=0,
    x_tick_sig_digits=4,
    max_xticks=None,
    x_scale='linear',
    symlog_linthresh=None,
):
    pretty = prettify_rows(rows)
    api.plot_metric(
        pretty,
        x=x,
        y='avg_test_rmse',
        hue='method',
        title=f'{title_base}: RMSE на тестовой части',
        xlabel=xlabel,
        ylabel='Средний RMSE на тестовой части',
        x_tick_rotation=x_tick_rotation,
        x_tick_sig_digits=x_tick_sig_digits,
        max_xticks=max_xticks,
        x_scale=x_scale,
        symlog_linthresh=symlog_linthresh,
    )
    api.plot_metric(
        pretty,
        x=x,
        y='avg_runtime_sec',
        hue='method',
        title=f'{title_base}: среднее время',
        xlabel=xlabel,
        ylabel='Среднее время, сек',
        x_tick_rotation=x_tick_rotation,
        x_tick_sig_digits=x_tick_sig_digits,
        max_xticks=max_xticks,
        x_scale=x_scale,
        symlog_linthresh=symlog_linthresh,
    )
    api.plot_metric(
        pretty,
        x=x,
        y='avg_iterations',
        hue='method',
        title=f'{title_base}: среднее число итераций',
        xlabel=xlabel,
        ylabel='Среднее число итераций',
        x_tick_rotation=x_tick_rotation,
        x_tick_sig_digits=x_tick_sig_digits,
        max_xticks=max_xticks,
        x_scale=x_scale,
        symlog_linthresh=symlog_linthresh,
    )


## 1. Базовый сценарий

Это общая точка отсчета для всех следующих экспериментов.

In [ ]:
base_scenario = research.make_scenario(
    'base_500x500_rank3',
    description='Базовый сценарий: 500x500, rank=3, случайные пропуски, слабый шум.',
    tags=['base', 'demo'],
    matrix=api.MatrixConfig(m=500, n=500, rank=3),
    structure=api.StructureConfig(mode='none'),
    missingness_field=api.MissingnessFieldConfig(mode='random'),
    mask_sampling=api.MaskSamplingConfig(observed_fraction=0.35, exact_fraction=True),
    split=api.SplitConfig(validation_fraction=0.15, test_fraction=0.15),
    noise=api.NoiseConfig(mode='gaussian', std=0.02),
)

{
    'name': base_scenario.name,
    'shape': f"{base_scenario.scenario.matrix.m} x {base_scenario.scenario.matrix.n}",
    'true_rank': base_scenario.scenario.matrix.rank,
    'noise_std': base_scenario.scenario.noise.std,
    'observed_fraction': base_scenario.scenario.mask_sampling.observed_fraction,
    'missingness_mode': base_scenario.scenario.missingness_field.mode,
}

## 2. Методы

Это общий набор методов для всех экспериментов в наборе.

In [ ]:
methods = [
    research.make_method(
        'soft_impute',
        label='Soft-Impute',
        key='soft',
        params={'max_iter': 60},
        bindings={'rank': research.make_binding('matrix.rank', cast='int')},
    ),
    research.make_method(
        'als',
        label='ALS',
        key='als',
        params={'max_iter': 60, 'init': 'spectral', 'reg': 1e-3},
        bindings={'rank': research.make_binding('matrix.rank', cast='int')},
    ),
    research.make_method(
        'compact_rgd',
        label='Compact RGD',
        key='compact_rgd',
        params={'max_iter': 80, 'init': 'spectral'},
        bindings={'rank': research.make_binding('matrix.rank', cast='int')},
    ),
    research.make_method(
        'compact_rgd_l2',
        label='Compact RGD + L2',
        key='compact_rgd_l2',
        params={'max_iter': 80, 'init': 'spectral', 'l2_reg': 0.03},
        bindings={'rank': research.make_binding('matrix.rank', cast='int')},
    ),
]

show_table([
    {
        'method': method.name,
        'label': method.label,
        'params': method.params,
    }
    for method in methods
], n=10)


## 3. Однофакторные эксперименты

Для числовых параметров можно не задавать `values` вручную: тогда берется дефолтная сетка.
Если хочется проверить конкретные точки, можно передать свой список `values=[...]`.

Важно: автоматическая сетка для таких экспериментов окончательно вычисляется после привязки к базовому сценарию через набор экспериментов.


In [ ]:
rank_experiment = research.make_experiment(
    'rank_sweep',
    description='Меняем только истинный ранг. Значения строятся автоматически.',
    vary_path='matrix.rank',
    vary_alias='rank',
)

noise_experiment = research.make_experiment(
    'noise_sweep',
    description='Меняем только уровень шума. Значения строятся автоматически.',
    vary_path='noise.std',
    vary_alias='noise',
)

missingness_experiment = research.make_experiment(
    'missingness_sweep',
    description='Меняем только тип пропусков.',
    vary_path='missingness_field.mode',
    vary_alias='missingness',
)

manual_rank_probe = research.make_experiment(
    'rank_manual_probe',
    description='Пример ручного набора значений ранга.',
    vary_path='matrix.rank',
    vary_alias='rank',
    values=[2, 4, 7, 12],
)

experiments = [rank_experiment, noise_experiment, missingness_experiment]

show_table([
    {
        'name': exp.name,
        'vary_path': exp.vary_path,
        'values_mode': 'manual' if exp.values else 'auto_from_base_scenario',
        'values_preview': exp.values if exp.values else '(будут вычислены после suite.resolved_experiments)',
    }
    for exp in experiments + [manual_rank_probe]
], n=10)


## 4. Набор экспериментов

Теперь у нас один объект, который знает:
- какой базовый сценарий использовать;
- какие методы запускать;
- какие однофакторные эксперименты надо прогнать.


In [ ]:
suite = research.make_suite(
    'core_research_suite',
    description='Набор однофакторных экспериментов от одного базового сценария.',
    tags=['demo', 'suite'],
    base_scenario=base_scenario,
    methods=methods,
    experiments=experiments,
    seeds=[41],
    output_dir='research_outputs/core_research_suite',
)

{
    'suite_name': suite.name,
    'n_experiments': len(suite.experiments),
    'n_methods': len(suite.methods),
    'seeds': suite.seeds,
}

## 5. Предпросмотр

Сначала убеждаемся, что будут запущены именно те сценарии, которые мы ожидаем.


In [ ]:
resolved_experiments = suite.resolved_experiments(PROJECT_DIR)
show_table([
    {
        'experiment': exp.name,
        'parameter': exp.vary_path,
        'values': exp.values,
        'n_methods': len(exp.methods),
    }
    for exp in resolved_experiments
], n=10)

In [ ]:
suite_preview = suite.preview(PROJECT_DIR)
show_table([
    {
        'experiment': row.get('experiment_name'),
        'scenario': row.get('scenario_name'),
        'rank': row.get('rank'),
        'noise': row.get('noise'),
        'missingness': row.get('missingness'),
    }
    for row in suite_preview
], n=12)

## 6. Запуск

Один `suite.run()` прогоняет все однофакторные эксперименты подряд.


In [ ]:
suite_run = suite.run(PROJECT_DIR)
print('experiment runs:', len(suite_run.experiment_runs))
print('manifest rows:', len(suite_run.manifest))
print('records:', len(suite_run.records))
print('summary rows:', len(suite_run.summary))

## 7. Перебор по рангу

Здесь интересуют три вещи: качество восстановления, время и число итераций.


In [ ]:
rank_summary = rows_for_experiment(suite_run.summary, 'rank_sweep')
plot_experiment_bundle(
    rank_summary,
    x='rank',
    xlabel='Ранг',
    title_base='Перебор по рангу',
    x_tick_rotation=0,
    max_xticks=7,
    x_scale='log',
)


## 8. Перебор по шуму

Для шума используем более компактные подписи по оси `x`, чтобы дробные значения читались нормально.


In [ ]:
noise_summary = rows_for_experiment(suite_run.summary, 'noise_sweep')
plot_experiment_bundle(
    noise_summary,
    x='noise',
    xlabel='Шум',
    title_base='Перебор по шуму',
    x_tick_rotation=12,
    x_tick_sig_digits=2,
    max_xticks=5,
    x_scale='symlog',
)


## 9. Перебор по типу пропусков

Категориальный эксперимент тоже удобно смотреть по тем же трем метрикам.


In [ ]:
missingness_summary = rows_for_experiment(suite_run.summary, 'missingness_sweep')
plot_experiment_bundle(
    missingness_summary,
    x='missingness',
    xlabel='Тип пропусков',
    title_base='Перебор по типу пропусков',
    x_tick_rotation=10,
    max_xticks=6,
)


## 10. Сохраняем базовый сценарий, набор экспериментов и результаты

То есть редактируем мы все в коде, а сохраняем уже в стандартный формат.


In [ ]:
saved_base = research.save_scenario(base_scenario, PROJECT_DIR)
saved_suite = research.save_suite(suite, PROJECT_DIR)
saved_run = suite_run.save()

{
    'base_scenario': saved_base,
    'suite': saved_suite,
    **saved_run,
}

## 11. Загружаем набор экспериментов обратно из файла

Проверяем, что сохраненный конфиг исследования воспроизводим.


In [ ]:
loaded_suite = research.load_suite(saved_suite)
show_table([
    {
        'experiment': exp.name,
        'parameter': exp.vary_path,
        'values': exp.values,
    }
    for exp in loaded_suite.resolved_experiments(PROJECT_DIR)
], n=10)